# Benchmark LLMs on Food Nutrition

Test LLMs (via OpenRouter) on estimating calories and macros without internet access.

In [1]:
import pandas as pd
import json, os
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI
from tqdm.auto import tqdm

In [2]:
DATA_DIR = Path('data')

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

## Load datasets

In [3]:
ds = pd.read_json(DATA_DIR / 'benchmark.json')
print(f'{len(ds)} examples loaded')
ds.head()

5000 examples loaded


,prompt,components,calories,protein_g,fat_g,carbs_g
0,a bowl of homemade potato chips,"[{'food_name': 'Potato chips, homemade, fried ...",90.9,1.6,3.0,15.3
1,a serving of pork and chicken chow mein,"[{'food_name': 'Chow mein, pork and chicken, h...",252.0,32.2,6.6,20.3
2,"440g of Lamb, shoulder, whole, roasted, lean a...","[{'food_name': 'Lamb, shoulder, whole, roasted...",1034.0,85.8,77.0,0.0
3,"420g of Chicken, whole, corn-fed, raw, meat an...","[{'food_name': 'Chicken, whole, corn-fed, raw,...",596.4,51.2,43.3,0.0
4,a portion of dry egg pasta,"[{'food_name': 'Pasta, egg, white, dried, raw'...",310.2,11.0,3.1,63.6


## Benchmark prompt & runner

In [4]:
BENCH_SYSTEM = """You are a nutrition expert. Given a food description, estimate the total nutritional content.

Return ONLY a JSON object with these keys:
- "calories": total kcal (number)
- "protein_g": grams of protein (number)
- "fat_g": grams of fat (number)  
- "carbs_g": grams of carbohydrates (number)

Be as accurate as possible. Do not explain, just return the JSON."""

In [5]:
def _call_model(model: str, prompt: str) -> dict:
    """Single LLM call, returns parsed prediction or Nones."""
    try:
        resp = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": BENCH_SYSTEM},
                {"role": "user", "content": prompt},
            ],
            temperature=0,
        )
        text = (resp.choices[0].message.content or "").strip()
        if not text:
            raise ValueError("Empty response")
        if text.startswith("```"):
            text = text.split("\n", 1)[1].rsplit("```", 1)[0].strip()
        return json.loads(text)
    except Exception as e:
        print(f"Error on '{prompt[:50]}': {e}")
        return {"calories": None, "protein_g": None, "fat_g": None, "carbs_g": None}


def run_benchmark(ds: pd.DataFrame, model: str, batch_size: int = 10, max_workers: int = 10) -> pd.DataFrame:
    """Run a model against every prompt, batch_size calls in parallel."""
    predictions = [None] * len(ds)
    prompts = ds['prompt'].tolist()
    
    with tqdm(total=len(ds), desc=model.split("/")[-1]) as pbar:
        for batch_start in range(0, len(prompts), batch_size):
            batch_end = min(batch_start + batch_size, len(prompts))
            batch_indices = range(batch_start, batch_end)
            
            with ThreadPoolExecutor(max_workers=max_workers) as executor:
                futures = {
                    executor.submit(_call_model, model, prompts[i]): i
                    for i in batch_indices
                }
                for future in as_completed(futures):
                    idx = futures[future]
                    predictions[idx] = future.result()
                    pbar.update(1)
    
    preds_df = pd.DataFrame(predictions)
    preds_df.columns = [f"pred_{c}" for c in preds_df.columns]
    
    return pd.concat([ds.reset_index(drop=True), preds_df], axis=1)

In [ ]:
MODELS = [
"openai/gpt-5-nano",
"openai/gpt-4o-mini",
]

## Run all models

In [7]:
RESULTS_DIR = DATA_DIR / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

results = {}
for model in MODELS:
    model_slug = model.split("/")[-1]
    out_path = RESULTS_DIR / f'{model_slug}.json'
    
    if out_path.exists():
        print(f"Skipping {model_slug} (already exists)")
        results[model] = pd.read_json(out_path)
        continue
    
    print(f"\nRunning {model}...")
    result = run_benchmark(ds, model)
    result.to_json(out_path, orient='records', indent=2)
    results[model] = result
    print(f"Saved to {out_path}")

print(f"\nDone! {len(results)} models benchmarked.")


Running anthropic/claude-sonnet-4.6...


claude-sonnet-4.6:   0%|          | 0/5000 [00:00<?, ?it/s]

Error on '490g of Milk bread, homemade': Expecting ',' delimiter: line 2 column 16 (char 17)
Error on '370g of Bannocks, made with beremeal, homemade': Expecting ',' delimiter: line 1 column 15 (char 14)
Error on '485g of Bread, naan, peshwari naan, takeaway and r': Expecting ',' delimiter: line 2 column 16 (char 17)
Error on '305g of Sesame seeds, 145g of Leeks, boiled in uns': Expecting ',' delimiter: line 2 column 16 (char 17)
Error on '455g of Pastries, cream filled, retail': Expecting ',' delimiter: line 1 column 15 (char 14)
Error on '320g of Beef, flank, pot-roasted, lean and fat': Expecting value: line 1 column 14 (char 13)
Error on '410g of Oregano, dried, ground': Expecting property name enclosed in double quotes: line 2 column 17 (char 18)
Error on '365g of Lentil cutlets, fried in vegetable oil, ho': Expecting ',' delimiter: line 2 column 16 (char 17)
Error on '425g of Beans, green, raw, 350g of Ham, 415g of Cu': Expecting ',' delimiter: line 2 column 16 (char 17)
Error on 

claude-haiku-4.5:   0%|          | 0/5000 [00:00<?, ?it/s]

Saved to data/results/claude-haiku-4.5.json

Done! 2 models benchmarked.
